# Bahrain Rent Prediction — Simplified Pipeline

**This notebook is a plain-language rewrite of `final_solution.ipynb`.** It runs
the exact same pipeline and produces the same predictions — only the presentation
changed:

- descriptive names (`oof_predictions` instead of `OOF`, `model_df` instead of `dm`)
- plain `for` loops instead of dense one-liners and dict-comprehension accumulation
- every step commented with *what* it does and *why*

**Task:** predict monthly rent (BHD) from propertyfinder.bh listings.
**Metric:** Mean Absolute Error (MAE). **This pipeline scored 86.5 on the leaderboard.**

Pipeline at a glance:

1. Load the data and drop the 17 rows with `rent > 10,000` (data errors)
2. Build ~200 features per listing (parsing, calendar, geography, title mining)
3. Learn leak-safe statistics *inside* each CV fold (target encodings, imputation)
4. Train 4 gradient boosters with fold bagging (5 folds × 3 seeds = 15 fits each)
5. Blend the 4 learners with optimized weights + calibrate the premium segment
6. Write the submission file

> Like the original, this notebook is not executed here: a full run takes
> ~1.5–2 h because of the 60+ model fits. All external API responses are cached
> on disk, so no network access is needed.


## 1. Setup

All heavy feature logic lives in the project module `feature_pipeline.py`; this
notebook only orchestrates it. The validated recipe (feature lists, rent cap,
smoothing strength) is stored in `final_config.json`.


In [ ]:
# --- standard library ---------------------------------------------------------
import json
import os
import time
import warnings

# --- third-party --------------------------------------------------------------
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import optuna
from catboost import CatBoostRegressor
from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

# --- project module ------------------------------------------------------------
# build_row_features : stateless per-row features (parsing, calendar, geo, title, ...)
# clean_target       : drops the data-error rows (rent above the cap)
# fit_stats          : learns fold statistics (target encodings, imputers) on TRAIN rows
# apply_stats        : attaches those learned statistics to any dataframe
# group_features     : returns the column names of a named feature group
# KEYWORD_VOCAB      : the top-100 title tokens used as one-hot flags
from feature_pipeline import (build_row_features, clean_target, fit_stats,
                              apply_stats, group_features, KEYWORD_VOCAB)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 200)

with open("final_config.json") as f:
    config = json.load(f)          # the validated recipe

SEEDS = (42, 1337, 7)   # three bagging seeds; averaging over seeds reduces split luck
N_SPLITS = 5            # 5-fold CV -> 5 folds x 3 seeds = 15 fits per learner
MAX_ROUNDS = 3000       # hard cap on boosting rounds ...
EARLY_STOP_ROUNDS = 70  # ... but stop early when validation MAE stops improving


## 2. Load & explore

10,578 train rows / 3,527 test rows. Known data-quality findings and how the
pipeline handles them:

| finding | action |
|---|---|
| 3 rows miss `rent` (same 3 rows miss Title/Area/Agent/Agency) | drop — a target can never be imputed |
| 17 rows with `rent > 10,000` (max is 400,000!) | drop — data errors (see §3) |
| `Amenities` ~3% missing | KNN imputation + missingness flag (done inside each fold) |
| `Availability_date` ~5% missing | median imputation + missingness flag (done inside each fold) |


In [ ]:
train_raw = pd.read_csv("data.csv")
test_raw = pd.read_csv("test.csv")
print(f"train {train_raw.shape} | test {test_raw.shape}")

# missing values, showing only the columns that have any
print(train_raw.isna().sum()[lambda s: s > 0])

# Quick look at the target. Rent has a heavy right tail and a few absurd values.
eda = train_raw.dropna(subset=["rent"]).copy()
eda["size_sqm"] = (eda["Size"].str.extract(r"/\s*([\d,]+)\s*sqm")[0]
                   .str.replace(",", "", regex=False).astype(float))
eda["rent_per_sqm"] = eda["rent"] / eda["size_sqm"]
print(eda["rent"].describe().round(1))

print("\ntop rents (data errors — e.g. a studio at 31,500 BHD):")
cols = ["Property_type", "Area", "Beds", "size_sqm", "rent", "rent_per_sqm"]
print(eda.nlargest(8, "rent")[cols].to_string())


In [ ]:
# The feature builder relies on these external-data caches (geocoded area
# coordinates + governorate statistics), so it runs fully offline.
print(pd.read_csv("area_coordinates.csv").head(3))
print(pd.read_csv("governorate_stats.csv"))


## 3. Build features & clean the target

`build_row_features(df)` is **stateless**: every value it produces depends only
on that row plus fixed external caches (area coordinates, POI lists, governorate
stats, known towers). That means train and test can be transformed independently
without any leakage. It creates ~200 features: parsed sizes/beds/baths, calendar
features, distances & POI counts, title keyword flags, and more.

**The outlier rule — drop exactly the 17 rows with `rent > 10,000`.** Dropping
*more* expensive rows keeps improving the cross-validation score (down to 77 MAE
at a 1,500 cap) but that is metric gaming: CV on a capped training set never
sees premium listings, while the leaderboard's test set contains 649 premium
villas. The honest check (train on capped rows, but *validate on all legitimate
rows*) proved every lower cap is worse — so we drop only the 17 clear errors.

**Online vs offline geocoding.** Every external response is cached on disk, so
the pipeline works without network. The code below loads `.env` and checks for a
`GOOGLE_MAPS_API_KEY`: if one is found, unseen areas/towers may be geocoded online
(`OFFLINE = False`); if the key is missing, the pipeline runs strictly from the
caches (`OFFLINE = True`).


In [ ]:
# Geocoding mode is decided by the environment: if .env loads a Google Maps
# API key, unseen areas/towers may be geocoded online (OFFLINE = False);
# without a key the pipeline runs strictly from the on-disk caches
# (OFFLINE = True).
from dotenv import load_dotenv

load_dotenv(".env")
GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY")
OFFLINE = not bool(GOOGLE_MAPS_API_KEY)
if OFFLINE:
    print("no Google Maps API key in .env -> OFFLINE mode (cached data only)")
else:
    print("Google Maps API key loaded from .env -> online geocoding enabled")

# The 3 rows without a Title are the same 3 rows without rent -> drop them first.
start = time.time()
train_features = build_row_features(train_raw.dropna(subset=["Title"]), offline=OFFLINE)
test_features = build_row_features(test_raw, offline=OFFLINE)
print(f"train features {train_features.shape} | test features {test_features.shape} "
      f"| {time.time() - start:.0f}s")

# Drop exactly the 17 data-error rows (rent > 10,000).
rent_cap = config["clean"]["rent_cap"]
model_df = clean_target(train_features, rent_cap=rent_cap)
model_df = model_df.reset_index(drop=True)   # tidy 0..N index keeps the CV loops simple
target = model_df["rent"].astype(float)
print(f"model rows: {len(model_df)} (dropped {len(train_features) - len(model_df)})")


## 4. Assemble the feature list

The final model uses **203 features (8 categorical)**:

- `config["num_feats"]` — the base numeric set from the validated recipe
  (sizes, ratios, amenity flags, distances, POI counts, target encodings, ...)
- calendar + availability/title-stat groups (`G1_calendar`, `G6_new`)
- 4 hand-made **interactions** (encoding × size/beds, navy flag × navy-base distance)
- 100 **keyword one-hot** flags (top title tokens — beat TF-IDF embeddings in testing)
- 7 **market** features (rent-per-sqm encodings, coast × sea-view, rotated coords)

Two small helpers prepare the data for the boosters:

- `prepare_model_matrix` — adds the interaction columns, selects the 203 columns,
  and normalizes dtypes (pandas nullable `boolean`/`Int64` confuse the boosters;
  missing categories become a real `"missing"` category).
- `align_categories_with_train` — gives validation/test data *exactly* the
  category sets of the train fold (an XGBoost requirement); categories the train
  fold never saw are mapped to `"missing"`.


In [ ]:
calendar_numeric, calendar_categorical = group_features("G1_calendar")
extra_numeric, _ = group_features("G6_new")   # availability + title stats + log(size)

interaction_features = ["te_area_x_size", "te_areapt_x_size",
                        "te_area_x_beds", "navy_x_dist"]
keyword_features = [f"kw_{word}" for word in KEYWORD_VOCAB]   # top-100 title tokens
market_features = ["te_area_ppsqm", "te_ppsqm_x_size", "te_areapt_ppsqm",
                   "te_subarea", "coast_x_seaview", "rot_xy_1", "rot_xy_2"]

NUMERIC_FEATURES = (config["num_feats"] + calendar_numeric + extra_numeric
                    + interaction_features + keyword_features + market_features)
CATEGORICAL_FEATURES = config["cat_feats"] + calendar_categorical
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# group-mean encodings that must be computed inside each fold (see section 5)
TARGET_ENCODINGS = config["te"] + ["te_subarea"]

print(f"{len(ALL_FEATURES)} features ({len(CATEGORICAL_FEATURES)} categorical)")


def add_interaction_features(df):
    """Four multiplicative interactions: encoding x size/beds, navy flag x distance."""
    out = df.copy()
    out["te_area_x_size"] = out["te_area"] * out["log_size_sqm"]
    out["te_areapt_x_size"] = out["te_area_ptype"] * out["log_size_sqm"]
    out["te_area_x_beds"] = out["te_area"] * out["beds_num"]
    out["navy_x_dist"] = out["navy_approved"] * out["dist_to_navy_base_km"]
    return out


def prepare_model_matrix(df):
    """Select the 203 modeling columns and normalize dtypes for the boosters."""
    X = add_interaction_features(df)[ALL_FEATURES].copy()

    # pandas nullable dtypes ("boolean", "Int64") confuse the boosters -> cast them
    for col in X.columns:
        if str(X[col].dtype) == "boolean":
            X[col] = X[col].astype("int8")
        elif str(X[col].dtype) == "Int64":
            X[col] = X[col].astype("float64")

    # boosters need a real "missing" category, not NaN
    for col in CATEGORICAL_FEATURES:
        X[col] = X[col].fillna("missing").astype("category")
        if "missing" not in X[col].cat.categories:
            X[col] = X[col].cat.add_categories("missing")
    return X


def align_categories_with_train(X, X_train):
    """Give X exactly the category sets of X_train.

    XGBoost requires validation/test categories to match the train fold's.
    Any category the train fold never saw is mapped to "missing".
    """
    X = X.copy()
    for col in CATEGORICAL_FEATURES:
        train_cats = X_train[col].cat.categories
        unseen_as_missing = X[col].where(X[col].isin(train_cats), "missing")
        X[col] = pd.Categorical(unseen_as_missing, categories=train_cats)
    return X


## 5. Leak-safe statistics

Some features *learn from other rows* — target encodings (average rent per
area/agency/...), rent-per-sqm encodings, listing counts, the Amenities KNN
imputer and the availability median. If these were computed on the whole dataset
before cross-validation, each validation row would partly influence its own
features (leakage) and CV scores would be optimistically biased.

So they are **fit on the train folds only** (`fit_stats`) and then *applied* to
the validation rows (`apply_stats`). Every CV loop below follows this pattern.


## 6. Tune XGBoost (short Optuna refinement)

XGBoost is the champion learner (~86% of the final blend weight), so it gets a
short hyperparameter refinement. The search is **warm-started** from the best
parameters found in the project's earlier modeling runs and only runs a handful
of trials — the 3 tuning folds are pre-computed once (with leak-safe statistics)
so each trial only pays for the model fits.


In [ ]:
# Pre-compute the three tuning folds once (leak-safe stats per fold).
tuning_folds = []
tuning_kfold = KFold(n_splits=3, shuffle=True, random_state=42)
for train_idx, valid_idx in tuning_kfold.split(model_df):
    fold_train = model_df.iloc[train_idx]
    fold_valid = model_df.iloc[valid_idx]

    fold_stats = fit_stats(fold_train, np.log1p(target.iloc[train_idx]),
                           te_names=TARGET_ENCODINGS, m_smooth=config["m_smooth"],
                           amenities_mode="knn", avail_impute=True)
    X_train = prepare_model_matrix(apply_stats(fold_train, fold_stats))
    X_valid = align_categories_with_train(
        prepare_model_matrix(apply_stats(fold_valid, fold_stats)), X_train)

    tuning_folds.append({
        "X_train": X_train,
        "y_train_log": np.log1p(target.iloc[train_idx]),
        "X_valid": X_valid,
        "y_valid": target.iloc[valid_idx],
    })


def xgboost_objective(trial):
    """Mean 3-fold MAE (in BHD) for one candidate set of XGBoost hyperparameters."""
    params = {
        "learning_rate":    trial.suggest_float("learning_rate", 0.008, 0.10, log=True),
        "max_depth":        trial.suggest_int("max_depth", 5, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 15.0, log=True),
        "subsample":        trial.suggest_float("subsample", 0.65, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 0.9),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    fold_maes = []
    for fold in tuning_folds:
        model = xgb.XGBRegressor(**params, n_estimators=MAX_ROUNDS,
                                 early_stopping_rounds=EARLY_STOP_ROUNDS,
                                 tree_method="hist", enable_categorical=True,
                                 eval_metric="mae", n_jobs=-1, random_state=42)
        # the model trains on log1p(rent); predictions are mapped back with expm1
        model.fit(fold["X_train"], fold["y_train_log"],
                  eval_set=[(fold["X_valid"], np.log1p(fold["y_valid"]))],
                  verbose=False)
        valid_pred_bhd = np.expm1(model.predict(fold["X_valid"]))
        fold_maes.append(mean_absolute_error(fold["y_valid"], valid_pred_bhd))
    return float(np.mean(fold_maes))


# best parameters from the earlier modeling runs — the search starts here
PREVIOUS_BEST_XGB_PARAMS = {
    "learning_rate": 0.01588339819938034, "max_depth": 9,
    "min_child_weight": 3.210692538133574, "subsample": 0.718148837463117,
    "colsample_bytree": 0.4983454551260783, "reg_alpha": 0.005991407906474037,
    "reg_lambda": 0.10375079637268099,
}

study = optuna.create_study(direction="minimize", study_name="xgb_final")
study.enqueue_trial(PREVIOUS_BEST_XGB_PARAMS)   # warm start from the known winner
study.optimize(xgboost_objective, n_trials=6)   # short refinement, not a full search
XGB_PARAMS = study.best_params
print(f"tuned 3-fold MAE: {study.best_value:.2f}")


## 7. Fold-bagged training of four learners

Four boosters, chosen for diversity across algorithm and target space:

| learner | algorithm | target | loss |
|---|---|---|---|
| `lgb_log` | LightGBM | log1p(rent) | L1 (= MAE) |
| `xgb_log` | XGBoost | log1p(rent) | squared (tuned) — the champion |
| `cb_log` | CatBoost | log1p(rent) | MAE |
| `lgbq`    | LightGBM | raw rent | quantile-0.5 (predicts the median directly) |

**Fold bagging:** each learner is trained 5 folds × 3 seeds = 15 times.

- Each fold's validation rows get an **out-of-fold (OOF) prediction** — together
  these give an honest estimate of the model and the raw material for blending.
- Each fit also predicts the test set; the final test prediction is the **average
  of all 15 fits**, which reduces variance from the random fold splits.

Why log1p(rent)? Rent is heavily right-skewed; training on the log scale keeps a
few expensive listings from dominating the loss. Predictions are mapped back to
BHD with `expm1` before scoring.


In [ ]:
# LightGBM with L1 loss (= MAE). Tuned in the project's earlier modeling runs.
LIGHTGBM_PARAMS = {
    "objective": "l1", "metric": "l1", "verbosity": -1, "n_jobs": -1,
    "max_bin": 127, "learning_rate": 0.06336060201608548, "num_leaves": 54,
    "min_child_samples": 11, "subsample": 0.9465438786450396,
    "colsample_bytree": 0.5447991030729846, "reg_alpha": 0.0075591206641849144,
    "reg_lambda": 0.0010591491747863968,
}

# CatBoost with MAE loss. Tuned in the project's earlier modeling runs.
CATBOOST_PARAMS = {
    "depth": 8, "learning_rate": 0.03520034718545651,
    "l2_leaf_reg": 5.511861002556324, "random_strength": 0.3193112232838608,
    "bagging_temperature": 0.5799947781153003,
}

LEARNER_NAMES = ["lgb_log", "xgb_log", "cb_log", "lgbq"]


def train_four_learners(X_train, y_train, X_valid, y_valid, X_test, seed):
    """Train the four base learners on one CV fold; predict validation and test.

    Returns two dicts {learner_name: predictions}, always in BHD (never log-space).
    """
    valid_preds = {}
    test_preds = {}

    y_train_log = np.log1p(y_train)
    y_valid_log = np.log1p(y_valid)

    # --- 1. LightGBM on log1p(rent) -----------------------------------------
    lgb_train = lgb.Dataset(X_train, y_train_log,
                            categorical_feature=CATEGORICAL_FEATURES)
    lgb_valid = lgb.Dataset(X_valid, y_valid_log,
                            categorical_feature=CATEGORICAL_FEATURES,
                            reference=lgb_train)
    lgb_model = lgb.train(LIGHTGBM_PARAMS, lgb_train, num_boost_round=MAX_ROUNDS,
                          valid_sets=[lgb_valid],
                          callbacks=[lgb.early_stopping(EARLY_STOP_ROUNDS,
                                                        verbose=False)])
    best_iter = lgb_model.best_iteration
    valid_preds["lgb_log"] = np.expm1(lgb_model.predict(X_valid, num_iteration=best_iter))
    test_preds["lgb_log"] = np.expm1(lgb_model.predict(X_test, num_iteration=best_iter))

    # --- 2. LightGBM quantile-0.5 on raw rent (predicts the median directly) --
    quantile_params = {**LIGHTGBM_PARAMS, "objective": "quantile", "alpha": 0.5}
    lgbq_train = lgb.Dataset(X_train, y_train,
                             categorical_feature=CATEGORICAL_FEATURES)
    lgbq_valid = lgb.Dataset(X_valid, y_valid,
                             categorical_feature=CATEGORICAL_FEATURES,
                             reference=lgbq_train)
    lgbq_model = lgb.train(quantile_params, lgbq_train, num_boost_round=MAX_ROUNDS,
                           valid_sets=[lgbq_valid],
                           callbacks=[lgb.early_stopping(EARLY_STOP_ROUNDS,
                                                         verbose=False)])
    best_iter = lgbq_model.best_iteration
    # rent can never be negative -> clip at 0
    valid_preds["lgbq"] = np.clip(lgbq_model.predict(X_valid, num_iteration=best_iter), 0, None)
    test_preds["lgbq"] = np.clip(lgbq_model.predict(X_test, num_iteration=best_iter), 0, None)

    # --- 3. XGBoost on log1p(rent) — the champion learner --------------------
    xgb_model = xgb.XGBRegressor(**XGB_PARAMS, n_estimators=MAX_ROUNDS,
                                 early_stopping_rounds=EARLY_STOP_ROUNDS,
                                 tree_method="hist", enable_categorical=True,
                                 eval_metric="mae", n_jobs=-1, random_state=seed)
    xgb_model.fit(X_train, y_train_log, eval_set=[(X_valid, y_valid_log)], verbose=False)
    valid_preds["xgb_log"] = np.expm1(xgb_model.predict(X_valid))
    test_preds["xgb_log"] = np.expm1(xgb_model.predict(X_test))

    # --- 4. CatBoost on log1p(rent) -------------------------------------------
    # CatBoost wants the categorical columns as plain strings.
    X_train_cb = X_train.copy()
    X_valid_cb = X_valid.copy()
    X_test_cb = X_test.copy()
    for col in CATEGORICAL_FEATURES:
        for X in (X_train_cb, X_valid_cb, X_test_cb):
            X[col] = X[col].astype(str)
    cat_column_indices = [X_train_cb.columns.get_loc(col) for col in CATEGORICAL_FEATURES]
    cb_model = CatBoostRegressor(loss_function="MAE", eval_metric="MAE",
                                 iterations=MAX_ROUNDS,
                                 early_stopping_rounds=EARLY_STOP_ROUNDS,
                                 random_seed=seed, verbose=False, **CATBOOST_PARAMS)
    cb_model.fit(X_train_cb, y_train_log, eval_set=(X_valid_cb, y_valid_log),
                 cat_features=cat_column_indices)
    valid_preds["cb_log"] = np.expm1(cb_model.predict(X_valid_cb))
    test_preds["cb_log"] = np.expm1(cb_model.predict(X_test_cb))

    return valid_preds, test_preds


In [ ]:
# Test features get their statistics from ALL training rows, computed once here.
# Inside the loop we only re-align their categories to each fold's category sets.
full_stats = fit_stats(model_df, np.log1p(target), te_names=TARGET_ENCODINGS,
                       m_smooth=config["m_smooth"], amenities_mode="knn",
                       avail_impute=True)
X_test_full = prepare_model_matrix(apply_stats(test_features, full_stats))

oof_runs = []   # one out-of-fold predictions DataFrame per seed, averaged at the end
n_train = len(model_df)
n_test = len(X_test_full)
test_pred_sum = pd.DataFrame(0.0, index=range(n_test), columns=LEARNER_NAMES)
n_fits_per_learner = len(SEEDS) * N_SPLITS   # test prediction = mean over all 15 fits

for seed in SEEDS:
    oof = pd.DataFrame(np.nan, index=range(n_train), columns=LEARNER_NAMES)
    kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    for fold_no, (train_idx, valid_idx) in enumerate(kfold.split(model_df), start=1):
        fold_train = model_df.iloc[train_idx]
        fold_valid = model_df.iloc[valid_idx]

        # everything learned from other rows is fit on the TRAIN rows only
        fold_stats = fit_stats(fold_train, np.log1p(target.iloc[train_idx]),
                               te_names=TARGET_ENCODINGS, m_smooth=config["m_smooth"],
                               amenities_mode="knn", avail_impute=True)

        X_train = prepare_model_matrix(apply_stats(fold_train, fold_stats))
        X_valid = align_categories_with_train(
            prepare_model_matrix(apply_stats(fold_valid, fold_stats)), X_train)
        X_test = align_categories_with_train(X_test_full.copy(), X_train)
        y_train = target.iloc[train_idx]
        y_valid = target.iloc[valid_idx]

        valid_preds, test_preds = train_four_learners(
            X_train, y_train, X_valid, y_valid, X_test, seed)

        for name in LEARNER_NAMES:
            oof.loc[valid_idx, name] = valid_preds[name]
            test_pred_sum[name] += test_preds[name]
        print(f"seed {seed:>4} | fold {fold_no}/{N_SPLITS} done", flush=True)

    oof_runs.append(oof)

# average OOF predictions across the 3 seeds; average test predictions across all fits
oof_predictions = oof_runs[0].copy()
for run in oof_runs[1:]:
    oof_predictions += run
oof_predictions /= len(oof_runs)
test_predictions = test_pred_sum / n_fits_per_learner

for name in LEARNER_NAMES:
    seed_maes = [mean_absolute_error(target, run[name]) for run in oof_runs]
    print(f"{name:8s} OOF MAE = {np.mean(seed_maes):.2f} ± {np.std(seed_maes):.2f}")


## 8. Blend the learners + premium calibration

**Blending:** the four learners are combined as a weighted average *in BHD space*
(the metric's space). The weights are optimized with SLSQP to minimize MAE,
constrained to be non-negative and sum to 1 (a convex combination — no
extrapolation). To keep the estimate honest, the weights are cross-validated:
learn them on 4/5 of the OOF rows, score the remaining 1/5.

**Premium calibration:** tree ensembles systematically *under*-predict expensive
listings (few premium training samples + median-like regression pulls toward the
middle). Wherever the blend predicts more than 900 BHD, we scale the prediction
by the median actual/predicted ratio on that segment (≈ ×1.02). The factor is
learned on the training part only inside the honest CV.


In [ ]:
def find_blend_weights(prediction_matrix, actual):
    """Convex weights (>= 0, summing to 1) that minimize the blend's MAE."""
    n_learners = prediction_matrix.shape[1]
    initial_weights = np.full(n_learners, 1.0 / n_learners)

    def blend_mae(weights):
        return mean_absolute_error(actual, prediction_matrix @ weights)

    result = minimize(blend_mae, initial_weights, method="SLSQP",
                      bounds=[(0.0, 1.0)] * n_learners,
                      constraints={"type": "eq", "fun": lambda w: w.sum() - 1.0})
    return result.x


def premium_calibration_factor(blend_pred, actual, threshold=900.0, min_rows=50):
    """Median actual/predicted ratio among rows predicted above `threshold`.

    Returns 1.0 (no correction) when there are too few premium rows to estimate
    the ratio reliably.
    """
    is_premium = blend_pred > threshold
    if is_premium.sum() < min_rows:
        return 1.0
    return float(np.median(actual[is_premium] / blend_pred[is_premium]))


oof_matrix = oof_predictions.to_numpy()
test_matrix = test_predictions.to_numpy()
actual = target.to_numpy()

# --- honest estimate: learn weights on 4/5 of OOF rows, score the rest --------
plain_maes = []
calibrated_maes = []
blend_kfold = KFold(n_splits=5, shuffle=True, random_state=999)
for train_idx, valid_idx in blend_kfold.split(oof_matrix):
    weights = find_blend_weights(oof_matrix[train_idx], actual[train_idx])
    valid_blend = oof_matrix[valid_idx] @ weights
    plain_maes.append(mean_absolute_error(actual[valid_idx], valid_blend))

    # calibrate: factor learned on the train part, applied to the validation part
    train_blend = oof_matrix[train_idx] @ weights
    factor = premium_calibration_factor(train_blend, actual[train_idx])
    calibrated_blend = valid_blend.copy()
    calibrated_blend[calibrated_blend > 900] *= factor
    calibrated_maes.append(mean_absolute_error(actual[valid_idx], calibrated_blend))

print(f"blend-CV plain {np.mean(plain_maes):.2f} | "
      f"calibrated {np.mean(calibrated_maes):.2f}")

# --- final weights + calibration factor, fitted on the full OOF ----------------
blend_weights = find_blend_weights(oof_matrix, actual)
oof_blend = oof_matrix @ blend_weights
calibration_factor = premium_calibration_factor(oof_blend, actual)
oof_calibrated = oof_blend.copy()
oof_calibrated[oof_calibrated > 900] *= calibration_factor

print("blend weights:",
      {name: round(float(w), 3) for name, w in zip(LEARNER_NAMES, blend_weights)})
print(f"final OOF: plain {mean_absolute_error(actual, oof_blend):.2f} | "
      f"calibrated {mean_absolute_error(actual, oof_calibrated):.2f} "
      f"(factor {calibration_factor:.4f})")


## 9. Submission

Blend the test predictions with the same weights, apply the same premium
calibration, clip at 0 (rent cannot be negative), and align the rows with the
official sample submission so the order and coverage are guaranteed correct.


In [ ]:
test_blend = test_matrix @ blend_weights
test_blend[test_blend > 900] *= calibration_factor

submission = pd.DataFrame({"Property_id": test_raw["Property_id"],
                           "rent": np.clip(test_blend, 0, None)})

# match the official sample submission exactly (row order + full coverage)
sample = pd.read_csv("sample_submission.csv")
submission = sample[["Property_id"]].merge(submission, on="Property_id", how="left")
assert submission["rent"].notna().all() and len(submission) == len(sample)

# NOTE: the original notebook writes "submission_v4.csv"; the simplified copy
# uses its own filename so a re-run can never clobber the original submission.
submission.to_csv("submission_simplified.csv", index=False)
print(f"submission_simplified.csv: {len(submission)} rows | "
      f"median {submission.rent.median():.0f} BHD")

# sanity check: the predicted distribution should shadow the actual one
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(np.log1p(actual), bins=60, alpha=0.6, density=True, label="train actual")
axes[0].hist(np.log1p(oof_calibrated), bins=60, alpha=0.6, density=True, label="OOF blend")
axes[0].hist(np.log1p(submission["rent"]), bins=60, alpha=0.6, density=True, label="test pred")
axes[0].legend()
axes[0].set_title("log1p(rent) distributions")
axes[1].scatter(actual, oof_calibrated, s=3, alpha=0.3)
axes[1].plot([0, 10_000], [0, 10_000], "r--")
axes[1].set_xlabel("actual")
axes[1].set_ylabel("OOF prediction")
plt.tight_layout()
plt.show()
